# 02 — Базова модель класифікації

## Мета

Передбачити один із трьох класів для наступних 5 торгових днів:

- `outperform`: дохідність MSFT перевищить SPY більш ніж на 1%;
- `neutral`: різниця буде в межах ±1%;
- `underperform`: MSFT відстане від SPY більш ніж на 1%.

Це **класифікація**, а не регресія, тому що готова ціль `target_class` є категоріальною. Регресію майбутньої надлишкової дохідності можна перевірити пізніше як окремий експеримент.

Test-період у цьому notebook не використовується для вибору моделі.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Magnitol110/MarketLens.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_path = Path("/content/MarketLens")
    if not repo_path.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)
    os.chdir(repo_path)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "README.md").exists(), f"Repository root not found: {ROOT}"

try:
    import sklearn
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-ml.txt")], check=True)

print("Repository:", ROOT)
print("Python:", sys.executable)

Repository: D:\GitHub\Tenerefe Group Project\MarketLens
Python: D:\GitHub\Tenerefe Group Project\MarketLens\.venv\Scripts\python.exe


## 1. Завантаження готових ознак

Використовуємо тільки `train` для навчання і `validation` для порівняння моделей. `test` лише рахуємо, але не відкриваємо його ціль і не робимо на ньому прогнозів.


In [2]:
import hashlib
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = ROOT / "data" / "processed" / "training_features.csv"
data = pd.read_csv(DATA_PATH, parse_dates=["date"])
feature_columns = [column for column in data.columns if column not in {"date", "split", "target_class"}]

train = data.loc[data["split"] == "train"].copy()
validation = data.loc[data["split"] == "validation"].copy()
test_row_count = int((data["split"] == "test").sum())

X_train = train[feature_columns]
y_train = train["target_class"]
X_validation = validation[feature_columns]
y_validation = validation["target_class"]

assert train["date"].max() < validation["date"].min()
assert not any("future" in column.lower() or "target" in column.lower() for column in feature_columns)
assert X_train.isna().sum().sum() == 0 and X_validation.isna().sum().sum() == 0

print("Train:", train["date"].min().date(), "—", train["date"].max().date(), len(train))
print("Validation:", validation["date"].min().date(), "—", validation["date"].max().date(), len(validation))
print("Untouched test rows:", test_row_count)
print("Features:", len(feature_columns))

Train: 2016-08-11 — 2023-07-05 1735
Validation: 2023-07-13 — 2024-12-26 368
Untouched test rows: 374
Features: 36


## 2. Фіксовані baseline-моделі

Без підбору параметрів перевіряємо:

1. `Dummy most frequent` — мінімальна контрольна точка.
2. `Logistic regression` — проста лінійна модель зі стандартизацією.
3. `Random forest` — обмежена нелінійна модель.
4. `Histogram gradient boosting` — сильніший табличний baseline.

Основна метрика — `macro_f1`, яка однаково враховує всі три класи.


In [3]:
RANDOM_STATE = 42

models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=6,
        min_samples_leaf=8,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        min_samples_leaf=20,
        l2_regularization=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

In [4]:
LABELS = ["underperform", "neutral", "outperform"]
results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_validation)
    predictions[name] = prediction
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_validation, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_validation, prediction),
        "macro_f1": f1_score(y_validation, prediction, average="macro"),
        "weighted_f1": f1_score(y_validation, prediction, average="weighted"),
    })

results_frame = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
display(results_frame.round(4))

best_name = results_frame.loc[0, "model"]
best_model = models[best_name]
best_prediction = predictions[best_name]
print("Selected baseline:", best_name)

                 model  accuracy  balanced_accuracy  macro_f1  weighted_f1
hist_gradient_boosting    0.3533             0.3574    0.3412       0.3400
   logistic_regression    0.3315             0.3355    0.3283       0.3273
         random_forest    0.3424             0.3524    0.3241       0.3189
   dummy_most_frequent    0.3641             0.3333    0.1780       0.1944
Selected baseline: hist_gradient_boosting


## 3. Confusion matrix найкращого baseline

Рядки — справжній клас, стовпці — прогнозований.


In [5]:
matrix = confusion_matrix(y_validation, best_prediction, labels=LABELS)
matrix_frame = pd.DataFrame(matrix, index=[f"true_{label}" for label in LABELS], columns=[f"pred_{label}" for label in LABELS])
display(matrix_frame)

per_class_f1 = f1_score(y_validation, best_prediction, labels=LABELS, average=None)
display(pd.DataFrame({"class": LABELS, "f1": per_class_f1}).round(4))

 pred_underperform  pred_neutral  pred_outperform
                24            39               57
                31            44               59
                13            39               62
       class     f1
underperform 0.2553
     neutral 0.3438
  outperform 0.4247


## 4. Збереження доказів

Зберігаємо локальний baseline pipeline і JSON із validation-метриками. Test залишається недоторканим для `04_evaluation.ipynb`.


In [6]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

MODEL_PATH = ROOT / "models" / "baseline_model.joblib"
METRICS_PATH = ROOT / "docs" / "baseline-metrics.json"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "model": best_model,
    "feature_columns": feature_columns,
    "labels": LABELS,
    "prediction_horizon_trading_days": 5,
}, MODEL_PATH)

metrics = {
    "selected_model": best_name,
    "selection_metric": "validation_macro_f1",
    "validation_period": {
        "first_date": validation["date"].min().date().isoformat(),
        "last_date": validation["date"].max().date().isoformat(),
        "rows": len(validation),
    },
    "test_rows_untouched": test_row_count,
    "training_data_sha256": sha256(DATA_PATH),
    "feature_count": len(feature_columns),
    "results": results_frame.to_dict(orient="records"),
    "confusion_matrix_labels": LABELS,
    "confusion_matrix": matrix.tolist(),
}
METRICS_PATH.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved model:", MODEL_PATH)
print("Saved metrics:", METRICS_PATH)
print("Test predictions made: 0")

Saved model: D:\GitHub\Tenerefe Group Project\MarketLens\models\baseline_model.joblib
Saved metrics: D:\GitHub\Tenerefe Group Project\MarketLens\docs\baseline-metrics.json
Test predictions made: 0


## Висновок

Baseline завершено, коли обрана модель перевищує Dummy за macro F1, метрики зафіксовані, а test не використовувався. Далі команда переглядає ознаки й можливий витік перед створенням фінального pipeline.
